In [ ]:
# ①ライブラリのインポート
from sklearn.ensemble import RandomForestClassifier
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import accuracy_score, roc_auc_score

import lightgbm as lgb

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# ② データ読み込み（正しいパス）
p = "/kaggle/input/competitions/playground-series-s6e4/"

df_train = pd.read_csv(p + "train.csv")
df_test = pd.read_csv(p + "test.csv")
df_submission = pd.read_csv(p + "sample_submission.csv")

df_train.head()


In [ ]:
# ③ データの基本確認
df_train.info()
df_train.describe()
df_train.isnull().sum()


In [ ]:
# ④ 目的変数と特徴量の分離
target = "Irrigation_Need"  # ← 必ず train.csv を見て正しい名前に変更

X = df_train.drop(columns=[target])
y = df_train[target]


In [ ]:
# ⑤.5 カテゴリ変数を Label Encoding（追加セル）

from sklearn.preprocessing import LabelEncoder

# X の中で dtype が object（文字列）の列を探す
cat_cols = X.select_dtypes(include=["object"]).columns

# 各カテゴリ列を Label Encoding
le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    df_test[col] = le.transform(df_test[col])
    le_dict[col] = le  # 必要なら保存


In [ ]:
from sklearn.model_selection import StratifiedKFold

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = []
fold_scores = []

for fold, (train_idx, valid_idx) in enumerate(kf.split(X, y)):
    print(f"Fold {fold+1}")

    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    model = RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    pred_valid = model.predict(X_valid)

    score = accuracy_score(y_valid, pred_valid)
    fold_scores.append(score)

    print(f"  Accuracy: {score:.5f}")

print("\nCV Mean:", np.mean(fold_scores))


In [ ]:
# ⑥ モデル定義

model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)


In [ ]:
# ⑦ 学習
model.fit(X_train, y_train)


In [ ]:
# ⑧ 検証スコアを確認

pred_valid = model.predict(X_valid)
accuracy_score(y_valid, pred_valid)


In [ ]:
# ⑨ テストデータで予測

pred_test = model.predict(df_test)


In [ ]:

# ⑩ 提出ファイル作成
df_submission["Irrigation_Need"] = pred_test
df_submission.to_csv("submission.csv", index=False)
df_submission.head()
